# Generate the first model based on feature importances provided by the Random Forest Regressor on 05_baseline_modeling

In [ ]:
import os
import sys
from glob import glob

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import keras
import keras_tuner as kt
import tensorflow as tf
import tensorboard as tb
import numpy as np
import pandas as pd

from src.loaders import load_data_array
from utils import create_submission, get_run_logdir

In [ ]:
def data_generator(sample_ids, features_subset):
    """Generator that yields one sample at a time"""
    wide_data = features_subset.values  # Your stats data
    
    for i, sample_id in enumerate(sample_ids):
        # Load one sample at a time
        X_deep_single, y_single = load_data_array([sample_id])
        
        # Convert to float32
        X_deep_single = X_deep_single[0].astype(np.float32)  # Remove batch dimension
        y_single = y_single[0].astype(np.float32)
        X_wide_single = wide_data[i].astype(np.float32)
        
        yield (X_wide_single, X_deep_single), y_single

features = pd.read_csv('../data/processed/final_features.csv')
is_train = features['split'] == 'train'

# Create datasets
train_ids = features.loc[is_train, 'sample_id'].reset_index(drop=True)
val_ids = features.loc[~is_train, 'sample_id'].reset_index(drop=True)

X_wide_train = features.loc[is_train].drop(columns=['sample_id', 'split', 'env_max', 'dom_freq'])
X_wide_val = features.loc[~is_train].drop(columns=['sample_id', 'split', 'env_max', 'dom_freq'])

# Create TensorFlow datasets
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_ids, X_wide_train),
    output_signature=(
        (tf.TensorSpec(shape=(6,), dtype=tf.float32),           # wide input
         tf.TensorSpec(shape=(5, 10001, 31), dtype=tf.float32)), # deep input  
        tf.TensorSpec(shape=(300, 1259), dtype=tf.float32)      # target
    )
)

val_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(val_ids, X_wide_val),
    output_signature=(
        (tf.TensorSpec(shape=(6,), dtype=tf.float32),
         tf.TensorSpec(shape=(5, 10001, 31), dtype=tf.float32)),
        tf.TensorSpec(shape=(300, 1259), dtype=tf.float32)
    )
)

# Optimize the pipeline
BATCH_SIZE = 8  # Start small due to large input size
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("✅ TensorFlow datasets created!")

In [ ]:
# Inputs
input_wide = keras.layers.Input(shape = [6], name='stats')
input_deep = keras.layers.Input(shape = [5, 10001, 31], name = 'gathers')

input_deep_flatten = keras.layers.Flatten()(input_deep)

x = input_deep_flatten

# Add 3 layers, each with n_neurons
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dense(32, activation='relu')(x)

# Concatenation
concatenation_layer = keras.layers.Concatenate()([input_wide, x])

# Final dense layer
hidden4 = keras.layers.Dense(377700)(concatenation_layer)

# Output reshape
output = keras.layers.Reshape([300, 1259])(hidden4)

# Create the model
model = keras.Model(inputs = [input_wide, input_deep], outputs = output)

model.compile(
    optimizer='adamw',
    loss='mse',
    metrics = ['mape', 'mae']
)

In [ ]:
model.summary()

In [ ]:
# Remember we need the following callbacks: tensoboard, reduce lr on plateau and model checkpoints
# Nevermind checkpoint will be used only when we get the best hyperparameters
log_dir = get_run_logdir(root_logdir='../logs/model02_mlp')

checkpoint_callback = keras.callbacks.ModelCheckpoint('../checkpoints/model_02_mlp.weights.h5', save_weights_only=True, save_best_only=True)
reduceLR_callback = keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5)
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=log_dir)

In [ ]:
model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,
    callbacks=[checkpoint_callback,reduceLR_callback,tensorboard_callback],   
)

In [ ]:
# Use the model to make predictions on the test set.
model.save('../models/model_02_mlp.keras')  # Saves architecture + weights
print("✅ Model saved to ../models/model_02_mlp.keras")